In [1]:
# 07_conformal_coverage.ipynb
# Pillar 2 / RQ4-RQ5 (inference, coverage): distribution-free prediction sets on PD,
# for logistic-WoE vs stump-boosting on GMSC. Split conformal with two scores:
#   - LAC (marginal): set = {labels with softmax prob >= 1-qhat}, marginal coverage.
#   - Mondrian (class-conditional): separate qhat per true class -> conditional coverage.
# Checks: (RQ4) empirical coverage hits nominal 1-alpha; set-size / interval width;
#         (RQ5) conditional coverage near the decision cutoff (borderline applicants).
# Also reports a conformal PD score-interval (log-odds +/- qhat -> sigmoid) as the
# 'PD prediction interval' the proposal asks for. 50 repeats for coverage distribution.
# Single-cell; greyscale seaborn; dpi600 png+pdf; no captions.

import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

SEED=42; N_REP=50; ALPHA=0.10         # nominal coverage 90%
rng=np.random.default_rng(SEED)
sns.set_theme(style='whitegrid', context='paper'); sns.set_palette('Greys')
plt.rcParams.update({'axes.edgecolor':'0.2','grid.color':'0.85','font.size':10})
ROOT=os.path.abspath(os.path.join(os.getcwd(),'..')); DATA=os.path.join(ROOT,'data')
TABLES=os.path.join(ROOT,'results','tables'); FIGS=os.path.join(ROOT,'results','figures')
def savefig(fig,name):
    for e in ('png','pdf'): fig.savefig(os.path.join(FIGS,name+'.'+e),dpi=600,bbox_inches='tight')
    plt.close(fig)

# ===================================================================
# 0. LOAD GMSC WoE design
# ===================================================================
woe_X=pd.read_parquet(os.path.join(DATA,'gmsc_woe.parquet'))
meta=json.load(open(os.path.join(DATA,'woe_maps.json')))
TARGET=meta['target']; FEATURES=meta['features']
y_all=woe_X[TARGET].values.astype(int); X_all=woe_X[FEATURES].values
print('GMSC WoE:',X_all.shape,'| bad rate %.4f'%y_all.mean(),'| nominal coverage %d%%'%int((1-ALPHA)*100))

# ===================================================================
# 1. CONFORMAL MACHINERY (binary classification, split conformal)
#    softmax over 2 classes from model prob p=P(bad):  probs=[1-p, p]
#    LAC score for true label y: s = 1 - prob[y].  qhat = ceil((n+1)(1-a))/n quantile.
#    Prediction set at test: {k : prob[k] >= 1-qhat}.
# ===================================================================
def lac_qhat(prob_cal, y_cal, alpha):
    s = 1.0 - prob_cal[np.arange(len(y_cal)), y_cal]
    n=len(s); k=int(np.ceil((n+1)*(1-alpha))); k=min(k,n)
    return np.sort(s)[k-1]
def mondrian_qhat(prob_cal, y_cal, alpha):
    q={}
    for cls in (0,1):
        m=y_cal==cls; s=1.0-prob_cal[m][:,cls]; n=m.sum()
        k=int(np.ceil((n+1)*(1-alpha))); k=min(k,n); q[cls]=np.sort(s)[k-1]
    return q
def pred_sets(prob_te, qhat):
    # qhat scalar (LAC) or dict (Mondrian); returns boolean (n,2) membership
    if isinstance(qhat, dict):
        thr=np.array([1-qhat[0], 1-qhat[1]])
    else:
        thr=np.array([1-qhat, 1-qhat])
    return prob_te >= thr[None,:]
def coverage_and_size(sets, y_te):
    cov=sets[np.arange(len(y_te)), y_te].mean()
    size=sets.sum(1).mean()
    return float(cov), float(size)

def model_prob(kind, Xtr,ytr,Xte, seed):
    if kind=='logit':
        m=LogisticRegression(max_iter=1000).fit(Xtr,ytr); p=m.predict_proba(Xte)[:,1]
    else:
        m=GradientBoostingClassifier(max_depth=1,learning_rate=0.3,n_estimators=300,
                                     random_state=seed).fit(Xtr,ytr); p=m.predict_proba(Xte)[:,1]
    return np.column_stack([1-p, p]), p

# ===================================================================
# 2. REPEATED train/cal/test, both models, LAC + Mondrian
# ===================================================================
rows=[]; cutoff_rows=[]; width_rows=[]
t0=time.time()
for rep in range(N_REP):
    Xtr,Xtmp,ytr,ytmp=train_test_split(X_all,y_all,test_size=0.4,stratify=y_all,random_state=SEED+rep)
    Xcal,Xte,ycal,yte=train_test_split(Xtmp,ytmp,test_size=0.5,stratify=ytmp,random_state=SEED+rep)
    for kind in ('logit','stump'):
        if kind=='logit':
            mdl=LogisticRegression(max_iter=1000).fit(Xtr,ytr)
        else:
            mdl=GradientBoostingClassifier(max_depth=1,learning_rate=0.3,n_estimators=300,
                                           random_state=SEED+rep).fit(Xtr,ytr)
        pc=mdl.predict_proba(Xcal)[:,1]; prob_cal=np.column_stack([1-pc,pc])
        p_te=mdl.predict_proba(Xte)[:,1]; prob_te=np.column_stack([1-p_te,p_te])
        # LAC
        q=lac_qhat(prob_cal,ycal,ALPHA); S=pred_sets(prob_te,q); cov,size=coverage_and_size(S,yte)
        rows.append([rep,kind,'LAC',cov,size])
        # Mondrian
        qm=mondrian_qhat(prob_cal,ycal,ALPHA); Sm=pred_sets(prob_te,qm); covm,sizem=coverage_and_size(Sm,yte)
        rows.append([rep,kind,'Mondrian',covm,sizem])
        # class-conditional coverage (Mondrian)
        for cls in (0,1):
            mcls=yte==cls; c=Sm[mcls][:,cls].mean()
            cutoff_rows.append([rep,kind,'class%d'%cls,float(c)])
        # RQ5: conditional coverage near cutoff (PD in [0.4,0.6] of its own range -> borderline)
        # use score percentile band 45-55% as 'borderline'
        lo,hi=np.percentile(p_te,[45,55]); band=(p_te>=lo)&(p_te<=hi)
        if band.sum()>0:
            cb=Sm[band][np.arange(band.sum()), yte[band]].mean()
            cutoff_rows.append([rep,kind,'borderline',float(cb)])
        # conformal PD score-interval width (log-odds +- qhat_score -> sigmoid)
        # regression-style: residual |logit_score - 0/1|? Instead use symmetric score band
        eta=np.log(np.clip(p_te,1e-6,1-1e-6)/np.clip(1-p_te,1e-6,1-1e-6))
        # calibrate half-width from calibration log-odds abs residual to label
        eta_cal=np.log(np.clip(prob_cal[:,1],1e-6,1-1e-6)/np.clip(prob_cal[:,0],1e-6,1-1e-6))
        resid=np.abs(eta_cal - (ycal*4-2))     # crude: target logit +-2 for 1/0
        nq=int(np.ceil((len(resid)+1)*(1-ALPHA))); nq=min(nq,len(resid)); hw=np.sort(resid)[nq-1]
        pd_lo=1/(1+np.exp(-(eta-hw))); pd_hi=1/(1+np.exp(-(eta+hw)))
        width_rows.append([rep,kind,float(np.mean(pd_hi-pd_lo))])
    if (rep+1)%10==0: print('  rep %d/%d (%.0fs)'%(rep+1,N_REP,time.time()-t0))
print('done %.0fs'%(time.time()-t0))

cov_df=pd.DataFrame(rows,columns=['rep','model','method','coverage','set_size'])
cond_df=pd.DataFrame(cutoff_rows,columns=['rep','model','stratum','coverage'])
width_df=pd.DataFrame(width_rows,columns=['rep','model','pd_interval_width'])

# ===================================================================
# 3. SUMMARY
# ===================================================================
summ=cov_df.groupby(['model','method']).agg(
    cov_mean=('coverage','mean'),cov_std=('coverage','std'),
    size_mean=('set_size','mean'),size_std=('set_size','std')).round(4)
print('\n=== Coverage & set-size (nominal %.2f) ==='%(1-ALPHA))
print(summ.to_string())
cond_summ=cond_df.groupby(['model','stratum']).agg(cov_mean=('coverage','mean'),cov_std=('coverage','std')).round(4)
print('\n=== Conditional coverage (Mondrian) ==='); print(cond_summ.to_string())
wsumm=width_df.groupby('model').agg(w_mean=('pd_interval_width','mean'),w_std=('pd_interval_width','std')).round(4)
print('\n=== Conformal PD interval width ==='); print(wsumm.to_string())

# ===================================================================
# 4. SAVE
# ===================================================================
cov_df.round(6).to_csv(os.path.join(TABLES,'07_coverage.csv'),index=False)
cond_df.round(6).to_csv(os.path.join(TABLES,'07_conditional_coverage.csv'),index=False)
width_df.round(6).to_csv(os.path.join(TABLES,'07_pd_interval_width.csv'),index=False)

# ===================================================================
# 5. FIGURE A — coverage vs nominal, by model x method
# ===================================================================
fig,ax=plt.subplots(figsize=(7.0,4.6))
groups=[('logit','LAC'),('stump','LAC'),('logit','Mondrian'),('stump','Mondrian')]
labels=['logit\nLAC','stump\nLAC','logit\nMondrian','stump\nMondrian']
data=[cov_df[(cov_df.model==m)&(cov_df.method==mt)]['coverage'].values for m,mt in groups]
bp=ax.boxplot(data,patch_artist=True,widths=0.6,medianprops=dict(color='0.0',lw=1.3))
for p,c in zip(bp['boxes'],['0.35','0.6','0.45','0.7']): p.set_facecolor(c); p.set_edgecolor('0.0')
ax.axhline(1-ALPHA,color='0.0',lw=1.2,ls='--',label='nominal %.2f'%(1-ALPHA))
ax.set_xticklabels(labels); ax.set_ylabel('empirical coverage')
ax.legend(frameon=True,facecolor='white',edgecolor='0.5'); savefig(fig,'07_coverage_box')

# ===================================================================
# 6. FIGURE B — conditional coverage (class0/class1/borderline), logit vs stump
# ===================================================================
fig,ax=plt.subplots(figsize=(7.0,4.6))
strata=['class0','class1','borderline']; xpos=np.arange(len(strata)); wdt=0.35
for i,(m,cl) in enumerate([('logit','0.35'),('stump','0.65')]):
    means=[cond_df[(cond_df.model==m)&(cond_df.stratum==s)]['coverage'].mean() for s in strata]
    errs =[cond_df[(cond_df.model==m)&(cond_df.stratum==s)]['coverage'].std() for s in strata]
    ax.bar(xpos+(i-0.5)*wdt,means,wdt,yerr=errs,color=cl,edgecolor='0.0',capsize=3,label=m)
ax.axhline(1-ALPHA,color='0.0',lw=1.2,ls='--'); ax.set_xticks(xpos); ax.set_xticklabels(['good(class0)','bad(class1)','borderline'])
ax.set_ylabel('conditional coverage'); ax.set_ylim(0.7,1.0)
ax.legend(frameon=True,facecolor='white',edgecolor='0.5'); savefig(fig,'07_conditional_coverage')

print('\nSaved figures: 07_coverage_box, 07_conditional_coverage')
print('Saved tables : 07_coverage, 07_conditional_coverage, 07_pd_interval_width')

# ===================================================================
# 7. RQ4/RQ5 HEADLINE
# ===================================================================
print('\n================ RQ4/RQ5 HEADLINE (nominal %.2f) ================'%(1-ALPHA))
for m in ('logit','stump'):
    lac=cov_df[(cov_df.model==m)&(cov_df.method=='LAC')]
    mon=cov_df[(cov_df.model==m)&(cov_df.method=='Mondrian')]
    print('%-6s: LAC cov=%.3f size=%.2f | Mondrian cov=%.3f size=%.2f'%(
        m,lac.coverage.mean(),lac.set_size.mean(),mon.coverage.mean(),mon.set_size.mean()))
bl=cond_df[cond_df.stratum=='borderline']
print('Borderline (RQ5) coverage: logit=%.3f, stump=%.3f'%(
    bl[bl.model=='logit'].coverage.mean(), bl[bl.model=='stump'].coverage.mean()))
print('PD interval width: logit=%.4f, stump=%.4f'%(
    width_df[width_df.model=='logit'].pd_interval_width.mean(),
    width_df[width_df.model=='stump'].pd_interval_width.mean()))
print('Reading: both reach nominal coverage (RQ4); Mondrian equalizes class-conditional')
print('coverage; borderline coverage shows cutoff-region reliability (RQ5).')
print('\n07 complete.')


GMSC WoE: (150000, 7) | bad rate 0.0668 | nominal coverage 90%


  rep 10/50 (64s)


  rep 20/50 (128s)


  rep 30/50 (193s)


  rep 40/50 (259s)


  rep 50/50 (325s)
done 325s

=== Coverage & set-size (nominal 0.90) ===
                cov_mean  cov_std  size_mean  size_std
model method                                          
logit LAC         0.8999   0.0016     0.9426    0.0022
      Mondrian    0.9004   0.0024     1.3365    0.0145
stump LAC         0.9001   0.0015     0.9418    0.0019
      Mondrian    0.9008   0.0027     1.3350    0.0166

=== Conditional coverage (Mondrian) ===
                  cov_mean  cov_std
model stratum                      
logit borderline    0.9817   0.0049
      class0        0.9004   0.0025
      class1        0.9007   0.0097
stump borderline    0.9817   0.0043
      class0        0.9008   0.0027
      class1        0.9007   0.0094

=== Conformal PD interval width ===
       w_mean   w_std
model                
logit  0.3546  0.0045
stump  0.3523  0.0050



Saved figures: 07_coverage_box, 07_conditional_coverage
Saved tables : 07_coverage, 07_conditional_coverage, 07_pd_interval_width

================ RQ4/RQ5 HEADLINE (nominal 0.90) ================
logit : LAC cov=0.900 size=0.94 | Mondrian cov=0.900 size=1.34
stump : LAC cov=0.900 size=0.94 | Mondrian cov=0.901 size=1.33
Borderline (RQ5) coverage: logit=0.982, stump=0.982
PD interval width: logit=0.3546, stump=0.3523
Reading: both reach nominal coverage (RQ4); Mondrian equalizes class-conditional
coverage; borderline coverage shows cutoff-region reliability (RQ5).

07 complete.
